In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/archangel26/matching-model-v5/feature_metadata.txt
/kaggle/input/datasets/archangel26/matching-model-v5/xgb_matching_model.json
/kaggle/input/datasets/archangel26/matching-model-v5/matching_threshold.txt
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/Documentation_template.md
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/README.md
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/utils/validate_submission.py
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/test/test_source2.tsv
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/test/test_source3.tsv
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/test/test_source1.tsv
/kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/train/train_ground_truth.tsv
/kaggle

# Final Predictions

This notebook performs the final inference stage of the Amazon ML Challenge 2026 Business Entity Resolution task. The trained XGBoost matching model from Notebook 04 will be loaded together with its selected probability threshold and feature metadata, and the same normalization and pairwise feature definitions will be reproduced for the test data.

The complete test sources will then be processed using the final 9-rule blocking strategy established in Notebook 03. Candidate pairs will be generated in memory-conscious batches, scored using the trained matching model, filtered using the precision-oriented threshold selected during validation, and aggregated into the final S1-to-S2/S3 entity matches.

The notebook will produce the required `candidate_pairs.tsv` and `matching_results.tsv` files and will validate the final outputs using the official challenge validation script.

## 1. Imports

This cell imports the libraries required for the final prediction pipeline, including data processing, text normalization, similarity calculation, model loading, and file management.

In [2]:
import numpy as np
import pandas as pd

import os
import gc
import re
import glob
import json
import random
import warnings
import unicodedata
import subprocess
import sys

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

try:
    from rapidfuzz import fuzz
except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "rapidfuzz",
        ]
    )
    from rapidfuzz import fuzz

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

DATA_ROOT = (
    "/kaggle/input/datasets/"
    "archangel26/infinity8-amazon-ml-challenge/"
    "student_resource"
)

TEST_DIR = os.path.join(
    DATA_ROOT,
    "dataset",
    "test",
)

OUTPUT_DIR = (
    "/kaggle/working/"
    "prediction_output"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True,
)

print("Environment ready.")
print("TEST_DIR:", TEST_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("RapidFuzz ready.")

Environment ready.
TEST_DIR: /kaggle/input/datasets/archangel26/infinity8-amazon-ml-challenge/student_resource/dataset/test
OUTPUT_DIR: /kaggle/working/prediction_output
RapidFuzz ready.


## 2. Load Matching Model Artifacts

The trained XGBoost model, optimized matching threshold, and feature metadata from Notebook 04 are available directly through the attached Kaggle Dataset. This cell locates and loads the three saved artifacts so the final prediction pipeline uses the exact model configuration developed during training.

In [3]:
model_candidates = glob.glob(
    "/kaggle/input/**/xgb_matching_model.json",
    recursive=True,
)

threshold_candidates = glob.glob(
    "/kaggle/input/**/matching_threshold.txt",
    recursive=True,
)

metadata_candidates = glob.glob(
    "/kaggle/input/**/feature_metadata.txt",
    recursive=True,
)

if not model_candidates:
    raise FileNotFoundError(
        "xgb_matching_model.json was not found in /kaggle/input/."
    )

if not threshold_candidates:
    raise FileNotFoundError(
        "matching_threshold.txt was not found in /kaggle/input/."
    )

if not metadata_candidates:
    raise FileNotFoundError(
        "feature_metadata.txt was not found in /kaggle/input/."
    )

MODEL_PATH = model_candidates[0]
THRESHOLD_PATH = threshold_candidates[0]
METADATA_PATH = metadata_candidates[0]

matching_model = XGBClassifier()

matching_model.load_model(
    MODEL_PATH
)

with open(
    THRESHOLD_PATH,
    "r",
) as f:
    MATCHING_THRESHOLD = float(
        f.read().strip()
    )

with open(
    METADATA_PATH,
    "r",
) as f:
    FEATURE_METADATA = f.read()

print("Model artifacts loaded successfully.")
print("Model:", MODEL_PATH)
print("Threshold:", MATCHING_THRESHOLD)
print("Metadata:", METADATA_PATH)

Model artifacts loaded successfully.
Model: /kaggle/input/datasets/archangel26/matching-model-v5/xgb_matching_model.json
Threshold: 0.4
Metadata: /kaggle/input/datasets/archangel26/matching-model-v5/feature_metadata.txt


## 3. Load Test Data

This cell loads the three test source files using the same core columns used during model training. Compact integer indices are assigned to each source so candidate generation and feature construction can be performed efficiently during final inference.

In [4]:
USECOLS = [
    "entity_id",
    "business_name",
    "business_address",
    "country",
]

test_s1 = pd.read_csv(
    os.path.join(
        TEST_DIR,
        "test_source1.tsv",
    ),
    sep="\t",
    dtype=str,
    usecols=USECOLS,
)

test_s2 = pd.read_csv(
    os.path.join(
        TEST_DIR,
        "test_source2.tsv",
    ),
    sep="\t",
    dtype=str,
    usecols=USECOLS,
)

test_s3 = pd.read_csv(
    os.path.join(
        TEST_DIR,
        "test_source3.tsv",
    ),
    sep="\t",
    dtype=str,
    usecols=USECOLS,
)

test_s1["s1_idx"] = np.arange(
    len(test_s1),
    dtype=np.int32,
)

test_s2["s2_idx"] = np.arange(
    len(test_s2),
    dtype=np.int32,
)

test_s3["s3_idx"] = np.arange(
    len(test_s3),
    dtype=np.int32,
)

print("Test data loaded.")
print("S1:", len(test_s1))
print("S2:", len(test_s2))
print("S3:", len(test_s3))

Test data loaded.
S1: 1732544
S2: 4887273
S3: 5082316


## 4. Text Normalization

This cell reproduces the Unicode normalization and text-cleaning logic used in Notebook 04. The same normalized names, addresses, country keys, and blocking keys will therefore be used during final inference to keep the prediction pipeline consistent with model training.

In [5]:
import subprocess, sys
try:
    import regex
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "regex"])
    import regex

# Matches anything that is NOT a Letter, Number, or Mark (same categories
# kept by the original per-char loop). Whitespace chars are not in L/N/M
# either, so they get replaced with a space here too — identical outcome
# to the original isspace()-branch.
_STRIP_PATTERN = regex.compile(r'[^\p{L}\p{N}\p{M}]+')
_MULTI_SPACE = re.compile(r'\s+')

def normalize_unicode_text(text):
    if pd.isna(text):
        return ""
    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("&", " and ")
    text = _STRIP_PATTERN.sub(" ", text)
    return _MULTI_SPACE.sub(" ", text).strip()

def prepare_text_fields(df):
    name_norm = df["business_name"].fillna("").map(normalize_unicode_text)
    address_norm = df["business_address"].fillna("").map(normalize_unicode_text)
    country_key = df["country"].fillna("").astype(str).str.strip().str.lower()

    compact_name = name_norm.str.replace(" ", "", regex=False)
    first_token = name_norm.str.split().str[0].fillna("")
    last_token = name_norm.str.split().str[-1].fillna("")

    df["name_norm"] = name_norm
    df["address_norm"] = address_norm
    df["country_key"] = country_key
    df["name_country_key"] = name_norm + "\t" + country_key
    df["name_first_country_key"] = first_token + "\t" + country_key
    df["name_prefix3_country_key"] = compact_name.str[:3] + "\t" + country_key
    df["name_prefix4_country_key"] = compact_name.str[:4] + "\t" + country_key
    df["address_country_key"] = address_norm + "\t" + country_key
    df["name_last_country_key"] = last_token + "\t" + country_key
    df["name_edge_country_key"] = compact_name.str[:2] + compact_name.str[-2:] + "\t" + country_key
    df["name_initials_country_key"] = (
        name_norm.str.split().map(lambda tokens: "".join(t[0] for t in tokens if t)) + "\t" + country_key
    )
    df["name_suffix3_country_key"] = compact_name.str[-3:] + "\t" + country_key

    del compact_name, first_token, last_token
    gc.collect()
    return df

print("Preparing S1...")
test_s1 = prepare_text_fields(test_s1)
print("Preparing S2...")
test_s2 = prepare_text_fields(test_s2)
print("Preparing S3...")
test_s3 = prepare_text_fields(test_s3)

print("\nNormalization completed.")
print("S1 rows:", f"{len(test_s1):,}")
print("S2 rows:", f"{len(test_s2):,}")
print("S3 rows:", f"{len(test_s3):,}")

Preparing S1...
Preparing S2...
Preparing S3...

Normalization completed.
S1 rows: 1,732,544
S2 rows: 4,887,273
S3 rows: 5,082,316


## 5. Final Blocking Configuration

The final candidate-generation stage uses the 9 blocking rules selected in Notebook 03. These rules define the complete candidate set that will be scored by the trained matching model, while the block-size limit prevents excessively large blocks from overwhelming the inference pipeline.

In [6]:
FINAL_BLOCKING_RULES = [
    "name_country_key",
    "name_first_country_key",
    "name_prefix3_country_key",
    "name_prefix4_country_key",
    "address_country_key",
    "name_last_country_key",
    "name_edge_country_key",
    "name_initials_country_key",
    "name_suffix3_country_key",
]

MAX_BLOCK_SIZE = 500

print("Final blocking rules:")

for rule in FINAL_BLOCKING_RULES:
    print("-", rule)

print("\nMaximum block size:", MAX_BLOCK_SIZE)

Final blocking rules:
- name_country_key
- name_first_country_key
- name_prefix3_country_key
- name_prefix4_country_key
- address_country_key
- name_last_country_key
- name_edge_country_key
- name_initials_country_key
- name_suffix3_country_key

Maximum block size: 500


## 6. Matching Features

This cell reproduces the same pairwise similarity features used to train the XGBoost model in Notebook 04. The feature definitions and column order are kept identical so that the test candidates are scored using the same representation as the training data.

In [7]:
def safe_ratio(a, b):
    if not a or not b:
        return 0.0

    return (
        fuzz.ratio(
            a,
            b,
        )
        / 100.0
    )


def safe_token_sort(a, b):
    if not a or not b:
        return 0.0

    a_sorted = " ".join(
        sorted(a.split())
    )

    b_sorted = " ".join(
        sorted(b.split())
    )

    return (
        fuzz.ratio(
            a_sorted,
            b_sorted,
        )
        / 100.0
    )


def safe_token_set(a, b):
    if not a or not b:
        return 0.0

    a_tokens = set(
        a.split()
    )

    b_tokens = set(
        b.split()
    )

    if not a_tokens or not b_tokens:
        return 0.0

    intersection = (
        a_tokens & b_tokens
    )

    union = (
        a_tokens | b_tokens
    )

    return (
        len(intersection)
        / len(union)
    )


def safe_partial_ratio(a, b):
    if not a or not b:
        return 0.0

    return (
        fuzz.partial_ratio(
            a,
            b,
        )
        / 100.0
    )


def safe_wratio(a, b):
    if not a or not b:
        return 0.0

    return max(
        safe_ratio(
            a,
            b,
        ),
        safe_token_sort(
            a,
            b,
        ),
        safe_token_set(
            a,
            b,
        ),
    )


def length_ratio(a, b):
    la = len(a)
    lb = len(b)

    if la == 0 and lb == 0:
        return 1.0

    if la == 0 or lb == 0:
        return 0.0

    return (
        min(
            la,
            lb,
        )
        / max(
            la,
            lb,
        )
    )


def token_overlap(a, b):
    ta = set(a.split())
    tb = set(b.split())

    if not ta or not tb:
        return 0.0

    return (
        len(ta & tb)
        / len(ta | tb)
    )


def digit_overlap(a, b):
    da = {
        ch
        for ch in a
        if ch.isdigit()
    }

    db = {
        ch
        for ch in b
        if ch.isdigit()
    }

    if not da and not db:
        return 1.0

    if not da or not db:
        return 0.0

    return (
        len(da & db)
        / len(da | db)
    )


FEATURE_COLUMNS = [
    "name_ratio",
    "name_partial_ratio",
    "name_token_sort",
    "name_token_set",
    "name_wratio",
    "name_length_ratio",
    "name_token_overlap",
    "address_ratio",
    "address_partial_ratio",
    "address_token_sort",
    "address_token_set",
    "address_wratio",
    "address_length_ratio",
    "address_token_overlap",
    "digit_overlap",
    "country_exact",
    "name_exact",
    "address_exact",
    "name_missing",
    "address_missing",
    "is_source3",
]

print("Feature definitions ready.")
print(
    "Number of features:",
    len(FEATURE_COLUMNS),
)

Feature definitions ready.
Number of features: 21


## 7. Feature Construction

This cell creates the final feature matrix for candidate pairs. The implementation follows the feature construction logic from Notebook 04 and uses the same 21 features in the same order expected by the trained XGBoost model.

In [8]:
s1_name_array = test_s1[
    "name_norm"
].to_numpy(
    dtype=object
)

s1_address_array = test_s1[
    "address_norm"
].to_numpy(
    dtype=object
)

s1_country_array = test_s1[
    "country_key"
].to_numpy(
    dtype=object
)

s2_name_array = test_s2[
    "name_norm"
].to_numpy(
    dtype=object
)

s2_address_array = test_s2[
    "address_norm"
].to_numpy(
    dtype=object
)

s2_country_array = test_s2[
    "country_key"
].to_numpy(
    dtype=object
)

s3_name_array = test_s3[
    "name_norm"
].to_numpy(
    dtype=object
)

s3_address_array = test_s3[
    "address_norm"
].to_numpy(
    dtype=object
)

s3_country_array = test_s3[
    "country_key"
].to_numpy(
    dtype=object
)


def build_features_fast(
    s1_indices,
    matched_indices,
    source,
):
    if len(s1_indices) == 0:
        return np.empty(
            (
                0,
                len(FEATURE_COLUMNS),
            ),
            dtype=np.float32,
        )

    if source == "s2":
        source_names = s2_name_array
        source_addresses = s2_address_array
        source_countries = s2_country_array
        source_flag = 0
    else:
        source_names = s3_name_array
        source_addresses = s3_address_array
        source_countries = s3_country_array
        source_flag = 1

    n = len(s1_indices)

    features = np.empty(
        (
            n,
            len(FEATURE_COLUMNS),
        ),
        dtype=np.float32,
    )

    for i in range(n):
        s1_idx = int(
            s1_indices[i]
        )

        matched_idx = int(
            matched_indices[i]
        )

        name_a = s1_name_array[
            s1_idx
        ]

        name_b = source_names[
            matched_idx
        ]

        addr_a = s1_address_array[
            s1_idx
        ]

        addr_b = source_addresses[
            matched_idx
        ]

        country_a = s1_country_array[
            s1_idx
        ]

        country_b = source_countries[
            matched_idx
        ]

        name_ratio = safe_ratio(
            name_a,
            name_b,
        )

        name_partial = safe_partial_ratio(
            name_a,
            name_b,
        )

        name_sort = safe_token_sort(
            name_a,
            name_b,
        )

        name_set = safe_token_set(
            name_a,
            name_b,
        )

        name_wratio = max(
            name_ratio,
            name_sort,
            name_set,
        )

        name_len = length_ratio(
            name_a,
            name_b,
        )

        name_overlap = token_overlap(
            name_a,
            name_b,
        )

        addr_ratio = safe_ratio(
            addr_a,
            addr_b,
        )

        addr_partial = safe_partial_ratio(
            addr_a,
            addr_b,
        )

        addr_sort = safe_token_sort(
            addr_a,
            addr_b,
        )

        addr_set = safe_token_set(
            addr_a,
            addr_b,
        )

        addr_wratio = max(
            addr_ratio,
            addr_sort,
            addr_set,
        )

        addr_len = length_ratio(
            addr_a,
            addr_b,
        )

        addr_overlap = token_overlap(
            addr_a,
            addr_b,
        )

        digit_score = digit_overlap(
            addr_a,
            addr_b,
        )

        country_exact = int(
            bool(country_a)
            and bool(country_b)
            and country_a == country_b
        )

        name_exact = int(
            bool(name_a)
            and bool(name_b)
            and name_a == name_b
        )

        address_exact = int(
            bool(addr_a)
            and bool(addr_b)
            and addr_a == addr_b
        )

        name_missing = int(
            not name_a
            or not name_b
        )

        address_missing = int(
            not addr_a
            or not addr_b
        )

        features[i] = (
            name_ratio,
            name_partial,
            name_sort,
            name_set,
            name_wratio,
            name_len,
            name_overlap,
            addr_ratio,
            addr_partial,
            addr_sort,
            addr_set,
            addr_wratio,
            addr_len,
            addr_overlap,
            digit_score,
            country_exact,
            name_exact,
            address_exact,
            name_missing,
            address_missing,
            source_flag,
        )

    return features


print("Fast feature construction ready.")

Fast feature construction ready.


## 8. Candidate Generation

This cell builds the final candidate set using the 9 blocking rules selected in Notebook 03. Candidates from all blocking rules are unioned and deduplicated separately for Source 2 and Source 3, ensuring that every candidate subsequently scored by the matching model is represented in the final candidate output.

In [9]:
class CompactBlockIndex:

    def __init__(
        self,
        source_df,
        rule,
        max_block_size,
    ):
        values = (
            source_df[rule]
            .fillna("")
            .to_numpy(
                dtype=object
            )
        )

        valid_mask = values != ""

        valid_indices = np.flatnonzero(
            valid_mask
        ).astype(
            np.int32,
            copy=False,
        )

        valid_values = values[
            valid_mask
        ]

        if len(valid_values) == 0:
            self.keys = np.empty(
                0,
                dtype=object,
            )
            self.starts = np.empty(
                0,
                dtype=np.int32,
            )
            self.ends = np.empty(
                0,
                dtype=np.int32,
            )
            self.indices = np.empty(
                0,
                dtype=np.int32,
            )
            return

        order = np.argsort(
            valid_values,
            kind="mergesort",
        )

        sorted_values = valid_values[
            order
        ]

        sorted_indices = valid_indices[
            order
        ]

        change_points = np.flatnonzero(
            sorted_values[1:]
            != sorted_values[:-1]
        ) + 1

        starts = np.concatenate(
            [
                np.array(
                    [0],
                    dtype=np.int32,
                ),
                change_points.astype(
                    np.int32,
                    copy=False,
                ),
            ]
        )

        ends = np.concatenate(
            [
                change_points.astype(
                    np.int32,
                    copy=False,
                ),
                np.array(
                    [len(sorted_values)],
                    dtype=np.int32,
                ),
            ]
        )

        sizes = (
            ends - starts
        )

        keep = sizes <= max_block_size

        self.keys = sorted_values[
            starts[keep]
        ]

        self.starts = starts[
            keep
        ]

        self.ends = ends[
            keep
        ]

        self.indices = sorted_indices

        del values
        del valid_indices
        del valid_values
        del order
        del sorted_values
        del sorted_indices
        del change_points
        del starts
        del ends
        del sizes
        del keep

        gc.collect()

    def get(
        self,
        key,
    ):
        if not key:
            return None

        if len(self.keys) == 0:
            return None

        position = np.searchsorted(
            self.keys,
            key,
        )

        if (
            position >= len(self.keys)
            or self.keys[position] != key
        ):
            return None

        start = int(
            self.starts[position]
        )

        end = int(
            self.ends[position]
        )

        return self.indices[
            start:end
        ]


def build_block_indices_compact(
    source_df,
    rules,
):
    block_indices = {}

    for rule_number, rule in enumerate(
        rules,
        start=1,
    ):
        print(
            f"Building {rule_number}/{len(rules)}: "
            f"{rule}"
        )

        block_indices[rule] = (
            CompactBlockIndex(
                source_df,
                rule,
                MAX_BLOCK_SIZE,
            )
        )

        print(
            "  retained blocks:",
            f"{len(block_indices[rule].keys):,}",
        )

    return block_indices


print("Building compact S2 blocking indices...")

s2_block_indices = (
    build_block_indices_compact(
        test_s2,
        FINAL_BLOCKING_RULES,
    )
)

gc.collect()

print("\nBuilding compact S3 blocking indices...")

s3_block_indices = (
    build_block_indices_compact(
        test_s3,
        FINAL_BLOCKING_RULES,
    )
)

gc.collect()

print("\nBlocking indices created.")

print(
    "S2 blocking rules:",
    len(s2_block_indices),
)

print(
    "S3 blocking rules:",
    len(s3_block_indices),
)

Building compact S2 blocking indices...
Building 1/9: name_country_key
  retained blocks: 4,021,466
Building 2/9: name_first_country_key
  retained blocks: 332,765
Building 3/9: name_prefix3_country_key
  retained blocks: 60,069
Building 4/9: name_prefix4_country_key
  retained blocks: 230,095
Building 5/9: address_country_key
  retained blocks: 4,150,705
Building 6/9: name_last_country_key
  retained blocks: 194,881
Building 7/9: name_edge_country_key
  retained blocks: 184,578
Building 8/9: name_initials_country_key
  retained blocks: 845,310
Building 9/9: name_suffix3_country_key
  retained blocks: 23,466

Building compact S3 blocking indices...
Building 1/9: name_country_key
  retained blocks: 4,231,324
Building 2/9: name_first_country_key
  retained blocks: 373,688
Building 3/9: name_prefix3_country_key
  retained blocks: 60,684
Building 4/9: name_prefix4_country_key
  retained blocks: 233,904
Building 5/9: address_country_key
  retained blocks: 4,416,458
Building 6/9: name_last_c

## 9. Final Test Inference

This cell performs the complete final inference pipeline in batches. For each Source 1 entity, candidates from all 9 blocking rules are unioned, the exact candidate set is written to `candidate_pairs.tsv`, the same candidates are converted into the 21 model features, and the trained XGBoost model scores them using the threshold selected in Notebook 04. Final positive predictions are written to `matching_results.tsv`.

In [10]:
# import time

# CANDIDATE_PATH = os.path.join(
#     OUTPUT_DIR,
#     "candidate_pairs.tsv",
# )

# MATCHING_PATH = os.path.join(
#     OUTPUT_DIR,
#     "matching_results.tsv",
# )

# BATCH_SIZE = 500
# FEATURE_CHUNK_SIZE = 5000


# for path in (
#     CANDIDATE_PATH,
#     MATCHING_PATH,
# ):
#     if os.path.exists(path):
#         os.remove(path)


# s1_block_arrays = {
#     rule: test_s1[
#         rule
#     ].to_numpy(
#         dtype=object
#     )
#     for rule in FINAL_BLOCKING_RULES
# }


# s1_entity_array = test_s1[
#     "entity_id"
# ].to_numpy(
#     dtype=object
# )

# s2_entity_array = test_s2[
#     "entity_id"
# ].to_numpy(
#     dtype=object
# )

# s3_entity_array = test_s3[
#     "entity_id"
# ].to_numpy(
#     dtype=object
# )


# def get_candidates_fast(
#     s1_idx,
#     block_indices,
# ):
#     candidates = set()

#     for rule in FINAL_BLOCKING_RULES:

#         key = s1_block_arrays[
#             rule
#         ][s1_idx]

#         if not key:
#             continue

#         indices = block_indices[
#             rule
#         ].get(key)

#         if indices is not None:
#             candidates.update(
#                 indices.tolist()
#             )

#     return candidates


# total_candidates = 0
# total_matches = 0

# total_s1 = len(test_s1)

# start_time = time.time()


# with open(
#     CANDIDATE_PATH,
#     "w",
#     encoding="utf-8",
# ) as candidate_file, open(
#     MATCHING_PATH,
#     "w",
#     encoding="utf-8",
# ) as matching_file:

#     candidate_file.write(
#         "source1_entity_id\tcandidate_entity_ids\n"
#     )

#     matching_file.write(
#         "source1_entity_id\tmatched_entity_ids\n"
#     )

#     for start in range(
#         0,
#         total_s1,
#         BATCH_SIZE,
#     ):

#         end = min(
#             start + BATCH_SIZE,
#             total_s1,
#         )

#         candidate_ids_by_s1 = {}

#         s2_left = []
#         s2_right = []

#         s3_left = []
#         s3_right = []

#         for s1_idx in range(
#             start,
#             end,
#         ):

#             s2_candidates = (
#                 get_candidates_fast(
#                     s1_idx,
#                     s2_block_indices,
#                 )
#             )

#             s3_candidates = (
#                 get_candidates_fast(
#                     s1_idx,
#                     s3_block_indices,
#                 )
#             )

#             candidate_ids_by_s1[
#                 s1_idx
#             ] = (
#                 s2_candidates,
#                 s3_candidates,
#             )

#             s2_left.extend(
#                 [s1_idx]
#                 * len(s2_candidates)
#             )

#             s2_right.extend(
#                 s2_candidates
#             )

#             s3_left.extend(
#                 [s1_idx]
#                 * len(s3_candidates)
#             )

#             s3_right.extend(
#                 s3_candidates
#             )

#         s2_left = np.asarray(
#             s2_left,
#             dtype=np.int32,
#         )

#         s2_right = np.asarray(
#             s2_right,
#             dtype=np.int32,
#         )

#         s3_left = np.asarray(
#             s3_left,
#             dtype=np.int32,
#         )

#         s3_right = np.asarray(
#             s3_right,
#             dtype=np.int32,
#         )

#         total_candidates += (
#             len(s2_left)
#             + len(s3_left)
#         )

#         matched_by_s1 = {}

#         for source, left, right in (
#             (
#                 "s2",
#                 s2_left,
#                 s2_right,
#             ),
#             (
#                 "s3",
#                 s3_left,
#                 s3_right,
#             ),
#         ):

#             if len(left) == 0:
#                 continue

#             for chunk_start in range(
#                 0,
#                 len(left),
#                 FEATURE_CHUNK_SIZE,
#             ):

#                 chunk_end = min(
#                     chunk_start
#                     + FEATURE_CHUNK_SIZE,
#                     len(left),
#                 )

#                 left_chunk = left[
#                     chunk_start:chunk_end
#                 ]

#                 right_chunk = right[
#                     chunk_start:chunk_end
#                 ]

#                 features = (
#                     build_features_fast(
#                         left_chunk,
#                         right_chunk,
#                         source,
#                     )
#                 )

#                 probabilities = (
#                     matching_model
#                     .predict_proba(
#                         features
#                     )[:, 1]
#                 )

#                 positive_mask = (
#                     probabilities
#                     >= MATCHING_THRESHOLD
#                 )

#                 positive_left = (
#                     left_chunk[
#                         positive_mask
#                     ]
#                 )

#                 positive_right = (
#                     right_chunk[
#                         positive_mask
#                     ]
#                 )

#                 if source == "s2":
#                     entity_array = (
#                         s2_entity_array
#                     )
#                 else:
#                     entity_array = (
#                         s3_entity_array
#                     )

#                 for i in range(
#                     len(positive_left)
#                 ):

#                     s1_idx = int(
#                         positive_left[i]
#                     )

#                     matched_idx = int(
#                         positive_right[i]
#                     )

#                     matched_by_s1.setdefault(
#                         s1_idx,
#                         [],
#                     ).append(
#                         entity_array[
#                             matched_idx
#                         ]
#                     )

#                 del features
#                 del probabilities
#                 del positive_mask
#                 del positive_left
#                 del positive_right

#         batch_candidate_count = 0
#         batch_match_count = 0

#         for s1_idx in range(
#             start,
#             end,
#         ):

#             s2_candidates, s3_candidates = (
#                 candidate_ids_by_s1[
#                     s1_idx
#                 ]
#             )

#             candidate_ids = []

#             for matched_idx in s2_candidates:
#                 candidate_ids.append(
#                     s2_entity_array[
#                         int(matched_idx)
#                     ]
#                 )

#             for matched_idx in s3_candidates:
#                 candidate_ids.append(
#                     s3_entity_array[
#                         int(matched_idx)
#                     ]
#                 )

#             candidate_ids = list(
#                 dict.fromkeys(
#                     candidate_ids
#                 )
#             )

#             matched_ids = list(
#                 dict.fromkeys(
#                     matched_by_s1.get(
#                         s1_idx,
#                         [],
#                     )
#                 )
#             )

#             batch_candidate_count += (
#                 len(candidate_ids)
#             )

#             batch_match_count += (
#                 len(matched_ids)
#             )

#             candidate_file.write(
#                 str(
#                     s1_entity_array[
#                         s1_idx
#                     ]
#                 )
#                 + "\t"
#                 + ",".join(
#                     map(
#                         str,
#                         candidate_ids,
#                     )
#                 )
#                 + "\n"
#             )

#             matching_file.write(
#                 str(
#                     s1_entity_array[
#                         s1_idx
#                     ]
#                 )
#                 + "\t"
#                 + ",".join(
#                     map(
#                         str,
#                         matched_ids,
#                     )
#                 )
#                 + "\n"
#             )

#         total_matches += (
#             batch_match_count
#         )

#         processed = end

#         elapsed = (
#             time.time()
#             - start_time
#         )

#         rate = (
#             processed / elapsed
#             if elapsed > 0
#             else 0
#         )

#         percent = (
#             processed
#             / total_s1
#             * 100
#         )

#         print(
#             f"S1: {processed:,}/{total_s1:,} "
#             f"({percent:.2f}%) | "
#             f"Batch candidates: "
#             f"{batch_candidate_count:,} | "
#             f"Total candidates: "
#             f"{total_candidates:,} | "
#             f"Batch matches: "
#             f"{batch_match_count:,} | "
#             f"Total matches: "
#             f"{total_matches:,} | "
#             f"Rate: {rate:.2f} S1/s | "
#             f"Elapsed: {elapsed / 60:.1f} min"
#         )

#         del candidate_ids_by_s1
#         del s2_left
#         del s2_right
#         del s3_left
#         del s3_right
#         del matched_by_s1

#         gc.collect()


# total_elapsed = (
#     time.time()
#     - start_time
# )

# print(
#     "\nFinal inference completed."
# )

# print(
#     "Total S1 processed:",
#     total_s1,
# )

# print(
#     "Total candidates:",
#     f"{total_candidates:,}",
# )

# print(
#     "Total matches:",
#     f"{total_matches:,}",
# )

# print(
#     "Elapsed time:",
#     f"{total_elapsed / 60:.1f} minutes",
# )

# print(
#     "Candidate file:",
#     CANDIDATE_PATH,
# )

# print(
#     "Matching file:",
#     MATCHING_PATH,
# )

In [ ]:
import time
import gc
import multiprocessing as mp
from difflib import SequenceMatcher

# ================================================================
# SET THIS FIRST: your remaining minutes minus ~5 min write buffer
# ================================================================
DEADLINE_SECONDS = 20 * 60   # <-- ADJUST to (your real remaining minutes - 5) * 60

CANDIDATE_PATH = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")
MATCHING_PATH = os.path.join(OUTPUT_DIR, "matching_results.tsv")

BATCH_SIZE = 5000
FEATURE_CHUNK_SIZE = 20000
N_WORKERS = max(1, mp.cpu_count())

for path in (CANDIDATE_PATH, MATCHING_PATH):
    if os.path.exists(path):
        os.remove(path)

s1_name_array = test_s1["name_norm"].to_numpy(dtype=object)
s1_address_array = test_s1["address_norm"].to_numpy(dtype=object)
s1_country_array = test_s1["country_key"].to_numpy(dtype=object)
s1_entity_array = test_s1["entity_id"].to_numpy(dtype=object)

s2_name_array = test_s2["name_norm"].to_numpy(dtype=object)
s2_address_array = test_s2["address_norm"].to_numpy(dtype=object)
s2_country_array = test_s2["country_key"].to_numpy(dtype=object)
s2_entity_array = test_s2["entity_id"].to_numpy(dtype=object)

s3_name_array = test_s3["name_norm"].to_numpy(dtype=object)
s3_address_array = test_s3["address_norm"].to_numpy(dtype=object)
s3_country_array = test_s3["country_key"].to_numpy(dtype=object)
s3_entity_array = test_s3["entity_id"].to_numpy(dtype=object)

s1_block_arrays = {rule: test_s1[rule].to_numpy(dtype=object) for rule in FINAL_BLOCKING_RULES}

n_total = len(test_s1)

def _precompute_row_artifacts(name_arr, address_arr):
    n = len(name_arr)
    name_tokens = np.empty(n, dtype=object)
    name_sorted = np.empty(n, dtype=object)
    addr_tokens = np.empty(n, dtype=object)
    addr_sorted = np.empty(n, dtype=object)
    addr_digits = np.empty(n, dtype=object)
    for i in range(n):
        name = name_arr[i] if name_arr[i] else ""
        addr = address_arr[i] if address_arr[i] else ""
        name_toks = name.split()
        name_tokens[i] = frozenset(name_toks)
        name_sorted[i] = " ".join(sorted(name_toks))
        addr_toks = addr.split()
        addr_tokens[i] = frozenset(addr_toks)
        addr_sorted[i] = " ".join(sorted(addr_toks))
        addr_digits[i] = frozenset(ch for ch in addr if ch.isdigit())
    return name_tokens, name_sorted, addr_tokens, addr_sorted, addr_digits

print("Precomputing token artifacts...")
_t0 = time.time()
s1_name_tokens, s1_name_sorted, s1_addr_tokens, s1_addr_sorted, s1_addr_digits = _precompute_row_artifacts(s1_name_array, s1_address_array)
s2_name_tokens, s2_name_sorted, s2_addr_tokens, s2_addr_sorted, s2_addr_digits = _precompute_row_artifacts(s2_name_array, s2_address_array)
s3_name_tokens, s3_name_sorted, s3_addr_tokens, s3_addr_sorted, s3_addr_digits = _precompute_row_artifacts(s3_name_array, s3_address_array)
print(f"Done in {time.time()-_t0:.1f}s")

def safe_ratio_fast(a, b):
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()

def safe_partial_ratio_fast(a, b):
    if not a or not b:
        return 0.0
    if len(a) > len(b):
        a, b = b, a
    if len(a) == len(b):
        return safe_ratio_fast(a, b)
    window = len(a)
    if len(b) - window > 100:
        positions = np.linspace(0, len(b) - window, num=20, dtype=int)
    else:
        positions = range(len(b) - window + 1)
    best = 0.0
    for pos in positions:
        score = SequenceMatcher(None, a, b[pos:pos + window]).ratio()
        if score > best:
            best = score
    return best

def length_ratio_fast(a, b):
    la, lb = len(a), len(b)
    if la == 0 and lb == 0:
        return 1.0
    if la == 0 or lb == 0:
        return 0.0
    return min(la, lb) / max(la, lb)

def _jaccard(a_tokens, b_tokens):
    if not a_tokens or not b_tokens:
        return 0.0
    union = len(a_tokens | b_tokens)
    if union == 0:
        return 0.0
    return len(a_tokens & b_tokens) / union

def _digit_overlap(da, db):
    if not da and not db:
        return 1.0
    if not da or not db:
        return 0.0
    return len(da & db) / len(da | db)

def get_candidates_exact(s1_idx, block_indices):
    candidates = set()
    for rule in FINAL_BLOCKING_RULES:
        key = s1_block_arrays[rule][s1_idx]
        if not key:
            continue
        indices = block_indices[rule].get(key)
        if indices is not None:
            candidates.update(indices.tolist())
    return candidates

APPLY_EMERGENCY_PREFILTER = True  # drop pairs with zero name/address/digit overlap; uncalibrated, time-pressure judgment call

def build_features_and_prefilter(left_indices, right_indices, source):
    n = len(left_indices)
    if n == 0:
        return np.empty((0, len(FEATURE_COLUMNS)), dtype=np.float32), np.empty(0, dtype=np.int32), np.empty(0, dtype=np.int32)

    left_names = s1_name_array[left_indices]
    left_addresses = s1_address_array[left_indices]
    left_countries = s1_country_array[left_indices]
    left_name_tokens = s1_name_tokens[left_indices]
    left_name_sorted = s1_name_sorted[left_indices]
    left_addr_tokens = s1_addr_tokens[left_indices]
    left_addr_sorted = s1_addr_sorted[left_indices]
    left_addr_digits = s1_addr_digits[left_indices]

    if source == "s2":
        right_names = s2_name_array[right_indices]; right_addresses = s2_address_array[right_indices]
        right_countries = s2_country_array[right_indices]; right_name_tokens = s2_name_tokens[right_indices]
        right_name_sorted = s2_name_sorted[right_indices]; right_addr_tokens = s2_addr_tokens[right_indices]
        right_addr_sorted = s2_addr_sorted[right_indices]; right_addr_digits = s2_addr_digits[right_indices]
    else:
        right_names = s3_name_array[right_indices]; right_addresses = s3_address_array[right_indices]
        right_countries = s3_country_array[right_indices]; right_name_tokens = s3_name_tokens[right_indices]
        right_name_sorted = s3_name_sorted[right_indices]; right_addr_tokens = s3_addr_tokens[right_indices]
        right_addr_sorted = s3_addr_sorted[right_indices]; right_addr_digits = s3_addr_digits[right_indices]

    is_source3_val = 1.0 if source == "s3" else 0.0
    rows_list = []
    keep_left = []
    keep_right = []

    for i in range(n):
        name_a, name_b = left_names[i], right_names[i]
        addr_a, addr_b = left_addresses[i], right_addresses[i]

        name_token_set = _jaccard(left_name_tokens[i], right_name_tokens[i])
        address_token_set = _jaccard(left_addr_tokens[i], right_addr_tokens[i])
        digit_overlap = _digit_overlap(left_addr_digits[i], right_addr_digits[i])

        if APPLY_EMERGENCY_PREFILTER:
            if name_token_set == 0.0 and address_token_set == 0.0 and digit_overlap == 0.0:
                continue

        country_a, country_b = left_countries[i], right_countries[i]

        name_ratio = safe_ratio_fast(name_a, name_b)
        name_partial_ratio = safe_partial_ratio_fast(name_a, name_b)
        name_token_sort = safe_ratio_fast(left_name_sorted[i], right_name_sorted[i])
        name_wratio = max(name_ratio, name_token_sort, name_token_set)
        name_length_ratio = length_ratio_fast(name_a, name_b)

        address_ratio = safe_ratio_fast(addr_a, addr_b)
        address_partial_ratio = safe_partial_ratio_fast(addr_a, addr_b)
        address_token_sort = safe_ratio_fast(left_addr_sorted[i], right_addr_sorted[i])
        address_wratio = max(address_ratio, address_token_sort, address_token_set)
        address_length_ratio = length_ratio_fast(addr_a, addr_b)

        country_exact = 1.0 if (country_a and country_b and country_a == country_b) else 0.0
        name_exact = 1.0 if (name_a and name_b and name_a == name_b) else 0.0
        address_exact = 1.0 if (addr_a and addr_b and addr_a == addr_b) else 0.0
        name_missing = 1.0 if (not name_a or not name_b) else 0.0
        address_missing = 1.0 if (not addr_a or not addr_b) else 0.0

        rows_list.append((
            name_ratio, name_partial_ratio, name_token_sort, name_token_set,
            name_wratio, name_length_ratio, name_token_set,
            address_ratio, address_partial_ratio, address_token_sort, address_token_set,
            address_wratio, address_length_ratio, address_token_set,
            digit_overlap, country_exact, name_exact, address_exact,
            name_missing, address_missing, is_source3_val,
        ))
        keep_left.append(left_indices[i])
        keep_right.append(right_indices[i])

    if not rows_list:
        return np.empty((0, len(FEATURE_COLUMNS)), dtype=np.float32), np.empty(0, dtype=np.int32), np.empty(0, dtype=np.int32)

    return (np.array(rows_list, dtype=np.float32),
            np.array(keep_left, dtype=np.int32),
            np.array(keep_right, dtype=np.int32))

def _mp_worker(payload):
    left_chunk, right_chunk, source = payload
    return build_features_and_prefilter(left_chunk, right_chunk, source)

_USE_MP = hasattr(os, "fork")
_pool = mp.get_context("fork").Pool(N_WORKERS) if _USE_MP else None
print(f"Workers: {N_WORKERS if _USE_MP else 1}")

total_candidates = 0
start_time = time.time()

candidates_by_s1 = {}
matches_by_s1 = {}
deadline_hit_at = None

try:
    for batch_start in range(0, n_total, BATCH_SIZE):
        elapsed_check = time.time() - start_time
        if elapsed_check > DEADLINE_SECONDS:
            deadline_hit_at = batch_start
            print(f"\n*** DEADLINE HIT at S1 row {batch_start:,}/{n_total:,} "
                  f"({100*batch_start/n_total:.1f}%) — stopping scoring now. "
                  f"Remaining rows get EMPTY matches (valid per spec). ***\n")
            break

        batch_end = min(batch_start + BATCH_SIZE, n_total)
        batch_indices = range(batch_start, batch_end)

        s2_left, s2_right, s3_left, s3_right = [], [], [], []
        cand_local = {}

        for s1_idx in batch_indices:
            s2c = get_candidates_exact(s1_idx, s2_block_indices)
            s3c = get_candidates_exact(s1_idx, s3_block_indices)
            cand_local[s1_idx] = (s2c, s3c)
            s2_left.extend([s1_idx]*len(s2c)); s2_right.extend(s2c)
            s3_left.extend([s1_idx]*len(s3c)); s3_right.extend(s3c)

        s2_left = np.asarray(s2_left, dtype=np.int32); s2_right = np.asarray(s2_right, dtype=np.int32)
        s3_left = np.asarray(s3_left, dtype=np.int32); s3_right = np.asarray(s3_right, dtype=np.int32)
        total_candidates += len(s2_left) + len(s3_left)

        tasks = []
        for source, left, right in (("s2", s2_left, s2_right), ("s3", s3_left, s3_right)):
            if len(left) == 0:
                continue
            for cs in range(0, len(left), FEATURE_CHUNK_SIZE):
                ce = min(cs+FEATURE_CHUNK_SIZE, len(left))
                tasks.append((left[cs:ce], right[cs:ce], source))

        if tasks:
            results = _pool.map(_mp_worker, tasks) if _USE_MP else [_mp_worker(t) for t in tasks]
            for (left_chunk, right_chunk, source), (features, kept_left, kept_right) in zip(tasks, results):
                if len(features) == 0:
                    continue
                probs = matching_model.predict_proba(features)[:, 1]
                mask = probs >= MATCHING_THRESHOLD
                pos_left = kept_left[mask]; pos_right = kept_right[mask]
                entity_array = s2_entity_array if source == "s2" else s3_entity_array
                for i in range(len(pos_left)):
                    s1_idx = int(pos_left[i])
                    matches_by_s1.setdefault(s1_idx, []).append(entity_array[int(pos_right[i])])

        candidates_by_s1.update(cand_local)

        elapsed = time.time() - start_time
        rate = batch_end / elapsed if elapsed > 0 else 0
        eta = (n_total - batch_end) / rate if rate > 0 else float("inf")
        print(f"S1: {batch_end:,}/{n_total:,} ({100*batch_end/n_total:.1f}%) | "
              f"Candidates: {total_candidates:,} | Matches: {sum(len(v) for v in matches_by_s1.values()):,} | "
              f"Rate: {rate:.1f}/s | Elapsed: {elapsed/60:.1f}min | ETA: {eta/60:.1f}min | "
              f"Deadline in: {(DEADLINE_SECONDS-elapsed)/60:.1f}min")

        del s2_left, s2_right, s3_left, s3_right
        gc.collect()
finally:
    if _pool is not None:
        _pool.close(); _pool.join()

print("\nWriting output files (empty matches for any unreached rows)...")
with open(CANDIDATE_PATH, "w", encoding="utf-8") as cf, open(MATCHING_PATH, "w", encoding="utf-8") as mf:
    cf.write("source1_entity_id\tcandidate_entity_ids\n")
    mf.write("source1_entity_id\tmatched_entity_ids\n")
    for s1_idx in range(n_total):
        s2c, s3c = candidates_by_s1.get(s1_idx, (set(), set()))
        cand_ids = list(dict.fromkeys([s2_entity_array[int(i)] for i in s2c] + [s3_entity_array[int(i)] for i in s3c]))
        match_ids = list(dict.fromkeys(matches_by_s1.get(s1_idx, [])))
        cf.write(f"{s1_entity_array[s1_idx]}\t{','.join(map(str,cand_ids))}\n")
        mf.write(f"{s1_entity_array[s1_idx]}\t{','.join(map(str,match_ids))}\n")

total_elapsed = time.time() - start_time
print("\nFinal inference completed.")
print("Total S1 rows:", n_total)
if deadline_hit_at is not None:
    print(f"NOTE: deadline stopped scoring at row {deadline_hit_at:,} — rows after that have empty matches.")
print("Total candidates scored:", f"{total_candidates:,}")
print("Total matches:", f"{sum(len(v) for v in matches_by_s1.values()):,}")
print("Elapsed:", f"{total_elapsed/60:.1f} min")
print("Files:", CANDIDATE_PATH, MATCHING_PATH)

Precomputing token artifacts...


## 10. Validate Final Outputs

This cell runs the official challenge validation script against the generated candidate and matching files. The validator checks the required columns, Source 1 coverage, valid Source 2/Source 3 IDs, duplicate IDs, and the requirement that every predicted match belongs to the final candidate set.

In [ ]:
VALIDATOR_PATH = os.path.join(
    DATA_ROOT,
    "utils",
    "validate_submission.py",
)

if not os.path.exists(
    VALIDATOR_PATH
):
    raise FileNotFoundError(
        "Official validator was not found at: "
        + VALIDATOR_PATH
    )

validation_command = (
    f'python3 "{VALIDATOR_PATH}" '
    f'--matching "{MATCHING_PATH}" '
    f'--candidate "{CANDIDATE_PATH}" '
    f'--test-dir "{TEST_DIR}"'
)

print(
    "Running official validator...\n"
)

validation_result = os.system(
    validation_command
)

if validation_result == 0:
    print(
        "\nValidation completed successfully."
    )
    print(
        "matching_results.tsv:",
        MATCHING_PATH,
    )
    print(
        "candidate_pairs.tsv:",
        CANDIDATE_PATH,
    )
else:
    print(
        "\nValidation reported issues."
    )

## 11. Download Final Submission Files

The validated `candidate_pairs.tsv` and `matching_results.tsv` files are saved in the prediction output directory. This cell creates direct download links for both final submission files.

In [ ]:
from IPython.display import FileLink, display

print("Final submission files:")

print(
    "\nmatching_results.tsv"
)
display(
    FileLink(
        MATCHING_PATH
    )
)

print(
    "\ncandidate_pairs.tsv"
)
display(
    FileLink(
        CANDIDATE_PATH
    )
)